In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
from torchvision.models import resnet18

model = resnet18()                        # 빈 ResNet18 (가중치 없이 모양만)
model.fc = nn.Linear(512, 2)              # Day 2처럼 마지막 층을 바꿔야 best.pth와 모양이 맞아요
model.load_state_dict(torch.load("best.pth"))
model.eval()                              # 내보내기 전에 꼭 "시험 시간"으로

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [2]:
dummy = torch.randn(1, 3, 224, 224)     # 견본 사진: 1장, 컬러 3, 세로, 가로

torch.onnx.export(
    model, dummy, "model.onnx",
    input_names=["image"],                # 입력 이름 (나중에 이 이름으로 사진을 넣어요)
    output_names=["scores"],              # 출력 이름
    dynamo=True,                          # 최신 내보내기 방식
    external_data=False,                  # 파일 하나에 전부 담기
)
print(f"model.onnx 크기: {os.path.getsize('model.onnx') / 1e6:.1f}MB")

W1026 20:18:59.837000 34456 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
model.onnx 크기: 44.8MB


In [3]:
assert os.path.exists("model.onnx"), "model.onnx 파일이 없어요"
print("통과! training 폴더에 model.onnx가 생겼어요")

통과! training 폴더에 model.onnx가 생겼어요


In [4]:
import onnxruntime as ort

sess = ort.InferenceSession("model.onnx")              # 연주자에게 악보 주기
onnx_out = sess.run(None, {"image": dummy.numpy()})[0]  # 입력 이름으로 사진 넣기

with torch.no_grad():
    torch_out = model(dummy).numpy()

print("ONNX   :", onnx_out)
print("PyTorch:", torch_out)
diff = np.abs(onnx_out - torch_out).max()
print("가장 큰 차이:", diff)

ONNX   : [[ 1.2765675  -0.99843854]]
PyTorch: [[ 1.2765672 -0.9984387]]
가장 큰 차이: 2.3841858e-07


In [5]:
assert diff < 1e-4, "결과가 너무 달라요"
print("통과! 악보가 원곡과 같아요")

통과! 악보가 원곡과 같아요


In [6]:
from torchvision.datasets import OxfordIIITPet
from torchvision import transforms as T
from predict import preprocess, predict          # 방금 만든 파일에서 함수 가져오기

raw_test = OxfordIIITPet(root="../data", split="test", target_types="binary-category")
test_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(),
                     T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

worst, correct, n = 0, 0, 0
for i in range(0, len(raw_test), 37):            # 37장 간격으로 약 100장
    img, label = raw_test[i]
    mine = preprocess(img)                        # 내가 만든 손질
    theirs = test_tf(img).unsqueeze(0).numpy()    # torchvision 손질
    worst = max(worst, np.abs(mine - theirs).max())

    prob = predict(img)
    answer = max(prob, key=prob.get)              # 확률이 더 큰 쪽
    correct += answer == ["고양이", "개"][label]
    n += 1

print(f"손질 결과의 가장 큰 차이: {worst}")
print(f"predict.py 정확도: {correct}/{n}")

손질 결과의 가장 큰 차이: 0
predict.py 정확도: 100/100


In [7]:
assert worst < 1e-4, "손질법이 달라요! predict.py의 빈칸을 다시 확인해 보세요"
print("통과! 손질법이 torchvision과 똑같아요")

통과! 손질법이 torchvision과 똑같아요


In [9]:
from PIL import Image
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"

img = Image.open("")              # training 폴더에 넣은 사진 파일 이름
prob = predict(img)

plt.imshow(img)
plt.title(f"고양이 {prob['고양이']:.0%} / 개 {prob['개']:.0%}")
plt.axis("off")
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: ''